# Population Benchmarks

This notebook implements the **population** benchmark programs from the appendix reduction for the synthetic experiments.

The full $m\times m$ regression matrix was reduced exactly to

$$
\mathbf{A}(\alpha,\beta)=\operatorname{diag}(\alpha,\beta \mathbf{I}_{m-1}),
$$

with $p=m=d/2$, $q=m-1$, $c=m^{-1/4}$, clean distribution $P$ as the law of $X\sim N(0,\mathbf{I}_m)$ and $Y=X+U$, and corrupted distribution $\tilde{P}$ as the law of $\tilde X=\operatorname{diag}(c,\mathbf{I}_q)X$.

The notebook includes:

1. CVXPY implementations of the constant-size DRO and MRO SDPs.
2. Closed-form ERM and SNR population benchmarks.
3. Associated plots for Setting A and Setting B (just for quick verification, not used in paper)

In [ ]:
# Core imports
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
print("installed CVX solvers:", cp.installed_solvers())

## Helpers and closed-form benchmarks

For the population experiment,

$$
R(\mathbf{A}(\alpha,\beta),P)=(\alpha-1)^2 + (m-1)(\beta-1)^2.
$$

ERM on the corrupted law has $\alpha=1/c$, $\beta=1$. SNR uses the appendix closed form with $\lambda=\rho^2$.


In [ ]:
@dataclass
class EstimatorResult:
    method: str
    m: int
    d: int
    rho: float
    c: float
    q: int
    alpha: float
    beta: float
    regret: float
    objective: float
    status: str
    solver: str
    solve_time: Optional[float] = None

def experiment_params(m: int):
    """Return q=m-1 and c=m^{-1/4}."""
    q = m - 1
    c = m ** (-0.25)
    return q, c

def clean_regret(alpha: float, beta: float, m: int) -> float:
    """Clean population regret R(A(alpha,beta), P)."""
    q, _ = experiment_params(m)
    return float((alpha - 1.0) ** 2 + q * (beta - 1.0) ** 2)

def corrupted_risk(alpha: float, beta: float, m: int) -> float:
    """Corrupted population risk L(A(alpha,beta), tilde P)."""
    q, c = experiment_params(m)
    return float((c * alpha - 1.0) ** 2 + q * ((beta - 1.0) ** 2 + 1.0))

def _corrupted_risk_expr(alpha, beta, q: int, c: float):
    """CVXPY expression for L(A(alpha,beta), tilde P)."""
    return cp.square(c * alpha - 1.0) + q * (cp.square(beta - 1.0) + 1.0)

def erm_population(m: int, rho: float = 1.0) -> EstimatorResult:
    q, c = experiment_params(m)
    alpha = 1.0 / c
    beta = 1.0
    return EstimatorResult(
        method="ERM",
        m=m,
        d=2 * m,
        rho=float(rho),
        c=float(c),
        q=q,
        alpha=float(alpha),
        beta=float(beta),
        regret=clean_regret(alpha, beta, m),
        objective=np.nan,
        status="closed_form",
        solver="closed_form",
    )

def snr_population(m: int, rho: float = 1.0, tol: float = 1e-12) -> EstimatorResult:
    q, c = experiment_params(m)
    lam = float(rho) ** 2
    threshold = c * (1.0 - c)
    if lam <= threshold + tol:
        alpha = c / (c**2 + lam)
        beta = 1.0
    else:
        t = (q + c) / (q + c**2 + lam)
        alpha = t
        beta = t
    return EstimatorResult(
        method="SNR",
        m=m,
        d=2 * m,
        rho=float(rho),
        c=float(c),
        q=q,
        alpha=float(alpha),
        beta=float(beta),
        regret=clean_regret(alpha, beta, m),
        objective=np.nan,
        status="closed_form",
        solver="closed_form",
    )


## CVXPY LMI blocks

The two LMI blocks are the appendix matrices $\mathcal Q_1(\alpha,\eta,t_1)$ and $\mathcal Q_b(\beta,\eta,T_b)$. They are affine in the CVXPY variables, so the resulting programs are SDPs.


In [ ]:
def _cvx_scalar(x):
    """Convert a scalar CVXPY expression or Python number into a 1x1 CVXPY block."""
    if not isinstance(x, cp.Expression):
        x = cp.Constant(float(x))
    return cp.reshape(x, (1, 1), order="F")

def Q1_block(alpha, eta, t1, c: float):
    """Return the 4x4 LMI block Q_1(alpha, eta, t_1)."""
    return cp.bmat([
        [_cvx_scalar(eta),     _cvx_scalar(0.0),  _cvx_scalar(alpha), _cvx_scalar(c * eta)],
        [_cvx_scalar(0.0),     _cvx_scalar(eta),  _cvx_scalar(-1.0),  _cvx_scalar(eta)],
        [_cvx_scalar(alpha),   _cvx_scalar(-1.0), _cvx_scalar(1.0),   _cvx_scalar(0.0)],
        [_cvx_scalar(c * eta), _cvx_scalar(eta),  _cvx_scalar(0.0),   _cvx_scalar(t1)],
    ])

def Qb_block(beta, eta, Tb):
    """Return the 5x5 LMI block Q_b(beta, eta, T_b)."""
    return cp.bmat([
        [_cvx_scalar(eta),  _cvx_scalar(0.0),  _cvx_scalar(beta), _cvx_scalar(eta),      _cvx_scalar(0.0)],
        [_cvx_scalar(0.0),  _cvx_scalar(eta),  _cvx_scalar(-1.0), _cvx_scalar(eta),      _cvx_scalar(eta)],
        [_cvx_scalar(beta), _cvx_scalar(-1.0), _cvx_scalar(1.0),  _cvx_scalar(0.0),      _cvx_scalar(0.0)],
        [_cvx_scalar(eta),  _cvx_scalar(eta),  _cvx_scalar(0.0),  _cvx_scalar(Tb[0, 0]), _cvx_scalar(Tb[0, 1])],
        [_cvx_scalar(0.0),  _cvx_scalar(eta),  _cvx_scalar(0.0),  _cvx_scalar(Tb[0, 1]), _cvx_scalar(Tb[1, 1])],
    ])

def solve_with_fallback(prob: cp.Problem, solver: Optional[str] = None, verbose: bool = False) -> str:
    """Solve an SDP with a reasonable solver fallback order."""
    installed = set(cp.installed_solvers())
    candidates = []
    if solver is not None:
        candidates.append(solver)
    for s in ["CLARABEL", "MOSEK", "CVXOPT", "SCS"]:
        if s in installed and s not in candidates:
            candidates.append(s)
    if not candidates:
        raise RuntimeError(
            "No SDP-capable solver found. Install CLARABEL, SCS, CVXOPT, or MOSEK. "
            f"Installed solvers are: {sorted(installed)}"
        )

    errors = []
    for s in candidates:
        try:
            if s == "SCS":
                prob.solve(solver=s, verbose=verbose, eps=1e-6, max_iters=100_000, warm_start=True)
            else:
                prob.solve(solver=s, verbose=verbose, warm_start=True)
        except Exception as exc:
            errors.append(f"{s}: {type(exc).__name__}: {exc}")
            continue
        if prob.status in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            return s
        errors.append(f"{s}: status={prob.status}")
    raise RuntimeError("All solver attempts failed:\n" + "\n".join(errors))

## DRO SDP

The population DRO program, derived in the appendix, is

$$
\begin{aligned}
\min_{\alpha,\beta,\eta,t_1,\mathbf{T}_b}\quad
&t_1+q\operatorname{tr}(\mathbf{T}_b)+\eta(\rho^2-c^2-1-3q)\\
\text{s.t.}\quad
&\mathcal Q_1(\alpha,\eta,t_1)\succeq \mathbf{0},\\
&\mathcal Q_b(\beta,\eta,\mathbf{T}_b)\succeq \mathbf{0},\\
&\eta\ge0, \quad \mathbf{T}_b \in \mathbb{S}^2.
\end{aligned}
$$


In [ ]:
def solve_dro_sdp(m: int, rho: float = 1.0, solver: Optional[str] = None, verbose: bool = False) -> EstimatorResult:
    q, c = experiment_params(m)

    alpha = cp.Variable(name="alpha")
    beta = cp.Variable(name="beta")
    eta = cp.Variable(name="eta")
    t1 = cp.Variable(name="t1")
    Tb = cp.Variable((2, 2), symmetric=True, name="Tb")

    constraints = [
        Q1_block(alpha, eta, t1, c) >> 0,
        Qb_block(beta, eta, Tb) >> 0,
        eta >= 0,
    ]
    objective = cp.Minimize(t1 + q * cp.trace(Tb) + eta * (rho**2 - c**2 - 1.0 - 3.0 * q))
    prob = cp.Problem(objective, constraints)
    used_solver = solve_with_fallback(prob, solver=solver, verbose=verbose)

    a = float(alpha.value)
    b = float(beta.value)
    return EstimatorResult(
        method="DRO",
        m=m,
        d=2 * m,
        rho=float(rho),
        c=float(c),
        q=q,
        alpha=a,
        beta=b,
        regret=clean_regret(a, b, m),
        objective=float(prob.value),
        status=prob.status,
        solver=used_solver,
        solve_time=getattr(prob.solver_stats, "solve_time", None),
    )


## MRO SDP

For MRO, define

$$
z_1=t_1-\eta,
\qquad
\mathbf{Z}_b = \mathbf{T}_b-\eta \mathbf 1_2\mathbf 1_2^\top.
$$

The resulting SDP, derived in the appendix, is

$$
\begin{aligned}
\min\quad
&\eta(\rho^2-c^2-q)+m\tau+y_1+q\operatorname{tr}(\mathbf{Y}_b)\\
\text{s.t.}\quad
&\mathcal Q_1(\alpha,\eta,t_1)\succeq \mathbf{0},\quad
\mathcal Q_b(\beta,\eta,\mathbf{T}_b) \succeq \mathbf{0},\quad
\eta\ge0,\\
&y_1\ge0,\quad y_1\ge z_1-\tau,\\
&\mathbf{Y}_b \succeq \mathbf{0},\quad \mathbf{Y}_b\succeq \mathbf{Z}_b-\tau \mathbf{I}_2,\\
&\mathbf{T}_b, \mathbf{Y}_b \in \mathbb{S}^2
\end{aligned}
$$


In [ ]:
def solve_mro_sdp(
    m: int,
    rho: float = 1.0,
    solver: Optional[str] = None,
    verbose: bool = False,
) -> EstimatorResult:
    q, c = experiment_params(m)

    alpha = cp.Variable(name="alpha")
    beta = cp.Variable(name="beta")
    eta = cp.Variable(name="eta")
    t1 = cp.Variable(name="t1")
    Tb = cp.Variable((2, 2), symmetric=True, name="Tb")
    tau = cp.Variable(name="tau")
    y1 = cp.Variable(name="y1")
    Yb = cp.Variable((2, 2), symmetric=True, name="Yb")

    oneone = np.ones((2, 2))
    I2 = np.eye(2)
    z1 = t1 - eta
    Zb = Tb - eta * oneone

    constraints = [
        Q1_block(alpha, eta, t1, c) >> 0,
        Qb_block(beta, eta, Tb) >> 0,
        eta >= 0,
        y1 >= 0,
        y1 >= z1 - tau,
        Yb >> 0,
        Yb - (Zb - tau * I2) >> 0,
    ]

    # Stage 1: solve the original MRO SDP.
    mro_objective = eta * (rho**2 - c**2 - q) + m * tau + y1 + q * cp.trace(Yb)
    prob = cp.Problem(cp.Minimize(mro_objective), constraints)
    used_solver = solve_with_fallback(prob, solver=solver, verbose=verbose)

    mro_value = float(prob.value)
    solve_time = getattr(prob.solver_stats, "solve_time", None)
    total_solve_time = 0.0 if solve_time is None else float(solve_time)

    # Stage 2: tie-break among numerically MRO-optimal solutions by corrupted risk.
    #
    # L(A(alpha,beta), tilde P) = (c alpha - 1)^2 + q((beta - 1)^2 + 1).
    # The tolerance is needed because conic solvers return approximate optima.
    mro_face_tol = 1e-6 * max(1.0, abs(mro_value))
    corrupted_risk_objective = cp.square(c * alpha - 1.0) + q * (
        cp.square(beta - 1.0) + 1.0
    )

    tie_prob = cp.Problem(
        cp.Minimize(corrupted_risk_objective),
        constraints + [mro_objective <= mro_value + mro_face_tol],
    )
    used_solver_tie = solve_with_fallback(tie_prob, solver=solver, verbose=verbose)

    tie_solve_time = getattr(tie_prob.solver_stats, "solve_time", None)
    if tie_solve_time is not None:
        total_solve_time += float(tie_solve_time)

    a = float(alpha.value)
    b = float(beta.value)

    return EstimatorResult(
        method="MRO",
        m=m,
        d=2 * m,
        rho=float(rho),
        c=float(c),
        q=q,
        alpha=a,
        beta=b,
        regret=clean_regret(a, b, m),
        objective=mro_value,
        status=f"{prob.status}; risk_tie_break={tie_prob.status}",
        solver=f"{used_solver}+{used_solver_tie}",
        solve_time=total_solve_time if total_solve_time > 0 else None,
    )

## Combined runner


In [ ]:
METHOD_ORDER = ["ERM", "DRO", "MRO", "SNR"]

def solve_all_estimators(
    m: int,
    rho: float = 1.0,
    solver: Optional[str] = None,
    verbose: bool = False,
    include_closed_forms: bool = True,
) -> pd.DataFrame:
    rows = []
    if include_closed_forms:
        rows.append(asdict(erm_population(m, rho)))
        rows.append(asdict(snr_population(m, rho)))
    rows.append(asdict(solve_dro_sdp(m, rho, solver=solver, verbose=verbose)))
    rows.append(asdict(solve_mro_sdp(m, rho, solver=solver, verbose=verbose)))
    df = pd.DataFrame(rows)
    df["method"] = pd.Categorical(df["method"], categories=METHOD_ORDER, ordered=True)
    return df.sort_values(["method"]).reset_index(drop=True)

def run_setting_A(
    d_values: Optional[Sequence[int]] = None,
    rho: float = 1.0,
    solver: Optional[str] = None,
    progress: bool = True,
) -> pd.DataFrame:
    """Setting A: vary d, with p=m=d/2 and estimator radius rho=1 by default."""
    if d_values is None:
        d_values = list(range(20, 401, 20))
    frames = []
    for d in d_values:
        if d % 2 != 0:
            raise ValueError(f"d must be even so that p=m=d/2; got d={d}.")
        if progress:
            print(f"Setting A: d={d}, m={d//2}, rho={rho:g}")
        frames.append(solve_all_estimators(d // 2, rho=rho, solver=solver))
    return pd.concat(frames, ignore_index=True)

def run_setting_B(
    d: int = 400,
    rho_values: Optional[Sequence[float]] = None,
    solver: Optional[str] = None,
    progress: bool = True,
) -> pd.DataFrame:
    """Setting B: fix d=400 and sweep rho=epsilon_hat."""
    if d % 2 != 0:
        raise ValueError(f"d must be even so that p=m=d/2; got d={d}.")
    if rho_values is None:
        rho_values = np.geomspace(0.1, 20.0, 50)
    m = d // 2
    frames = []
    for rho in rho_values:
        if progress:
            print(f"Setting B: d={d}, m={m}, rho={rho:.4g}")
        frames.append(solve_all_estimators(m, rho=float(rho), solver=solver))
    return pd.concat(frames, ignore_index=True)

## Setting A: vary dimension

Default grid: $d\in\{20,40,\dots,400\}$ with $\rho=1$.


In [ ]:
df_A = run_setting_A(d_values=list(range(20, 401, 20)), rho=1.0, progress=True)
df_A.head(12)


In [ ]:
def plot_setting_A(df: pd.DataFrame, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 4.5))
    for method in METHOD_ORDER:
        sub = df[df["method"] == method].sort_values("d")
        if len(sub) == 0:
            continue
        ax.plot(sub["d"], sub["regret"], marker="o", label=method)
    ax.set_xlabel("dimension d = 2m")
    ax.set_ylabel("clean population regret")
    ax.set_title("Setting A: population benchmark regrets")
    ax.legend()
    ax.grid(True, alpha=0.3)
    return ax

plot_setting_A(df_A)
plt.show()


## Setting B: sweep the supplied radius $\rho$

Default grid: $d=400$ and $\rho\in[0.1,20]$ on a log scale. The true corruption budget in the experiment is $\epsilon=1$.


In [ ]:
rho_grid = np.geomspace(0.1, 20.0, 50)
df_B = run_setting_B(d=400, rho_values=rho_grid, progress=True)
df_B.head(12)

In [ ]:
def plot_setting_B(df: pd.DataFrame, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 4.5))
    for method in METHOD_ORDER:
        sub = df[df["method"] == method].sort_values("rho")
        if len(sub) == 0:
            continue
        ax.plot(sub["rho"], sub["regret"], marker="o", markersize=3, label=method)
    ax.axvline(1.0, linestyle=":", label="true epsilon = 1")
    ax.set_xscale("log")
    ax.set_xlabel(r"supplied radius $\rho$")
    ax.set_ylabel("clean population regret")
    ax.set_title("Setting B: sensitivity to supplied radius")
    ax.legend()
    ax.grid(True, alpha=0.3)
    return ax

plot_setting_B(df_B)
plt.show()


## Save results

This cell writes CSVs and PDF plots into the current working directory.


In [ ]:
out_dir = Path.cwd()
df_A.to_csv(out_dir / "benchmarks_A.csv", index=False)
df_B.to_csv(out_dir / "benchmarks_B.csv", index=False)

print(f"Saved CSVs in: {out_dir.resolve()}")